# Chapter Title Generator
**Problem Statement:** Authors often struggle with coming up with a creative, innovative, and/or fitting title for the chapters of their book or even the book itself. This process can interrupt writing flow and may be particularly difficult for new writers. 

**Proposed Methodology:** We propose a supervised deep learning approach using transformer-based sequence-to-sequence text generation models to solve this problem. We plan to train a model to generate a list of suitable titles


In [1]:
import re
import pandas as pd
from datasets import load_dataset, concatenate_datasets
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

In [2]:
# --- Configuration & Utils ---

class Config:
    CHUNK_CHARS = 1000
    INPUT_PREFIX = "generate title: "
    ENABLE_PROFANITY_FILTER = True
    BLOCKED_WORDS = ["fuck", "shit", "bitch", "cunt", "slut", "ass", "dick"] 
    MAX_DATASET_SIZE = 100_000  # Cap the total dataset size to guarantee <8hr runtime

cfg = Config()

try:
    from better_profanity import profanity
    profanity.load_censor_words()
    _HAS_PROFANITY = True
except ImportError:
    _HAS_PROFANITY = False

_MAX_PROFANITY_CHARS = 512
_BLOCK_RE = re.compile(r"\b(" + "|".join(re.escape(w) for w in cfg.BLOCKED_WORDS) + r")\b", re.I)

def is_safe_text(text: str) -> bool:
    if not cfg.ENABLE_PROFANITY_FILTER or not text:
        return True
    if _BLOCK_RE.search(text):
        return False
    if _HAS_PROFANITY and len(text) <= _MAX_PROFANITY_CHARS:
        return not profanity.contains_profanity(text)
    return True

def clean_and_format(text: str) -> str:
    text = text.strip()
    truncated = text if len(text) <= cfg.CHUNK_CHARS else text[:cfg.CHUNK_CHARS]
    return cfg.INPUT_PREFIX + truncated

def is_descriptive_title(title: str) -> bool:
    title_lower = title.lower().strip()
    clean_text = re.sub(r'[^a-z\s]', ' ', title_lower)
    words = clean_text.split()
    
    if not words: 
        return False
        
    generic_keywords = {
        "chapter", "chapters", "scene", "scenes", "act", "acts", 
        "part", "parts", "book", "books", "prologue", "epilogue", 
        "volume", "volumes", "section", "sections", "story", "stories",
        "untitled", "i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"
    }
    
    if all(word in generic_keywords for word in words):
        return False
    return True

# --- Dataset Parsers ---

def process_cmu(batch):
    titles, texts, valid = [], [], []
    for title, summary in zip(batch["title"], batch["summary"]):
        title, summary = (title or "").strip(), (summary or "").strip()
        is_ok = (len(summary) >= 100 and 2 <= len(title) <= 120 and is_descriptive_title(title)
                 and is_safe_text(summary[:cfg.CHUNK_CHARS]) and is_safe_text(title))
        titles.append(title)
        texts.append(clean_and_format(summary) if is_ok else "")
        valid.append(is_ok)
    return {"title": titles, "text": texts, "is_valid": valid, "source": ["cmu_book_summaries"] * len(valid)}

def process_booksum(batch, use_summary=True):
    titles, texts, valid = [], [], []
    narrative_key = "summary_text" if use_summary else "chapter"
    for title, raw_narrative in zip(batch["summary_name"], batch[narrative_key]):
        title, narrative = (title or "").strip(), (raw_narrative or "").strip()
        is_ok = (len(narrative) >= 100 and 2 <= len(title) <= 120 and is_descriptive_title(title)
                 and is_safe_text(narrative[:cfg.CHUNK_CHARS]) and is_safe_text(title))
        titles.append(title)
        texts.append(clean_and_format(narrative) if is_ok else "")
        valid.append(is_ok)
    return {"title": titles, "text": texts, "is_valid": valid, "source": ["novel_chapter_booksum"] * len(valid)}

def process_gutenberg(batch):
    titles, texts, valid = [], [], []
    for raw_text in batch["text"]:
        raw_text = (raw_text or "").strip()
        match = re.match(r"^([A-Z0-9\s.,;:!?'-]+?)\n+(.*)", raw_text, flags=re.DOTALL)
        is_ok = False
        title, body = "", ""
        if match:
            title = match.group(1).strip().strip(";:.,-")
            body = match.group(2).strip()
            body_start = body.lower()[:300]
            is_metadata = ("author of" in body_start or "by " in body_start or 
                           "transcribed by" in body_start or "reprinted" in body_start or
                           re.match(r"^(no\.\s*\d+|vol\.\s*\d+|volume\s*\d+)", body_start))
            if (title.isupper() and len(title) > 2 and not is_metadata and 
                is_descriptive_title(title) and is_safe_text(body[:cfg.CHUNK_CHARS]) and is_safe_text(title)):
                is_ok = True
        titles.append(title)
        texts.append(clean_and_format(body) if is_ok else "")
        valid.append(is_ok)
    return {"title": titles, "text": texts, "is_valid": valid, "source": ["gutenberg_chapters"] * len(valid)}

In [3]:
#LOAD DATASETS
def pipeline_load_and_clean(dataset_id, process_fn, config_name=None, split="train"):
    print(f"Loading {dataset_id}...")
    ds = load_dataset(dataset_id, config_name, split=split) if config_name else load_dataset(dataset_id, split=split)
    processed = ds.map(process_fn, batched=True, remove_columns=ds.column_names)
    filtered = processed.filter(lambda x: x["is_valid"] is True).remove_columns(["is_valid"])
    print(f" -> Retained {len(filtered)} valid rows from {dataset_id}") 
    return filtered

if __name__ == "__main__":
    print("Loading and compiling datasets...")
    cmu_ds = pipeline_load_and_clean("textminr/cmu-book-summaries", process_cmu)
    booksum_ds = pipeline_load_and_clean("kmfoda/booksum", lambda b: process_booksum(b, use_summary=True))
    gutenberg_ds = pipeline_load_and_clean("zkeown/gutenberg-corpus", process_gutenberg, config_name="chapters")
    
    final_dataset = concatenate_datasets([cmu_ds, booksum_ds, gutenberg_ds]).shuffle(seed=42)
    
    # Cap dataset size for runtime control
    if len(final_dataset) > cfg.MAX_DATASET_SIZE:
        print(f"\nCapping dataset to {cfg.MAX_DATASET_SIZE} rows to ensure fast training...")
        final_dataset = final_dataset.select(range(cfg.MAX_DATASET_SIZE))
    
    print("\n--- Final Processed Dataset Preview (Shuffled) ---")
    display_df = pd.DataFrame(final_dataset.select(range(10)))
    print(display_df[["source", "title", "text"]])

Loading and compiling datasets...
Loading textminr/cmu-book-summaries...


Map:   0%|          | 0/16559 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16559 [00:00<?, ? examples/s]

 -> Retained 15665 valid rows from textminr/cmu-book-summaries
Loading kmfoda/booksum...


Map:   0%|          | 0/9600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9600 [00:00<?, ? examples/s]

 -> Retained 1235 valid rows from kmfoda/booksum
Loading zkeown/gutenberg-corpus...


Map:   0%|          | 0/305 [00:00<?, ? examples/s]

Filter:   0%|          | 0/305 [00:00<?, ? examples/s]

 -> Retained 115 valid rows from zkeown/gutenberg-corpus

--- Final Processed Dataset Preview (Shuffled) ---
               source                                              title  \
0  cmu_book_summaries  The Thirteen and a Half Lives of Captain Bluebear   
1  cmu_book_summaries                                  The October Horse   
2  cmu_book_summaries                                             Lilith   
3  cmu_book_summaries                                   The Green Knight   
4  cmu_book_summaries                               The Heidi Chronicles   
5  cmu_book_summaries                           Clear and Present Danger   
6  cmu_book_summaries                                        Hadji Murat   
7  cmu_book_summaries                             The Sterkarm Handshake   
8  cmu_book_summaries                                      The Magicians   
9  cmu_book_summaries                                      Lord of Chaos   

                                                text  

In [4]:
#TOKENIZER
print("\nInitializing T5 tokenizer...")
model_name = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(batch):
    inputs = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)
    targets = tokenizer(batch["title"], padding="max_length", truncation=True, max_length=32)
    
    labels = [
        [(label if label != tokenizer.pad_token_id else -100) for label in sequence] 
        for sequence in targets["input_ids"]
    ]
    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": labels
    }

print("Tokenizing dataset...")
tokenized_dataset = final_dataset.map(tokenize_function, batched=True, remove_columns=final_dataset.column_names)
split_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=42)


Initializing T5 tokenizer...
Tokenizing dataset...


Map:   0%|          | 0/17015 [00:00<?, ? examples/s]

In [5]:
#MODEL INITIALIZATION
print(f"\nLoading {model_name}...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


Loading google-t5/t5-small...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [6]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
        
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # T5 outputs might need minor cleanups for robust rouge scoring
    decoded_preds = ["\n".join(pred.strip().split()) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split()) for label in decoded_labels]
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

In [7]:
#TRAINING LOOPS
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
        output_dir="./t5-title-generator",
        eval_strategy="epoch",            
        logging_strategy="steps",
        logging_steps=100,
        save_strategy="epoch",            
        learning_rate=2e-4,               
        per_device_train_batch_size=8,    
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,    
        weight_decay=0.01,
        save_total_limit=1,
        num_train_epochs=3,               
        predict_with_generate=True,       
        report_to="none",
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        fp16=False
    )

print("\nInitializing Seq2SeqTrainer...")
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting training loop...")
trainer.train()

print("Training complete. Saving final model...")
trainer.save_model("./t5-title-generator-final")
tokenizer.save_pretrained("./t5-title-generator-final")


Initializing Seq2SeqTrainer...
Starting training loop...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,6.376261,2.935581,28.059100,11.599700,27.951900,28.076000
2,5.870678,2.863726,29.677100,12.740200,29.547900,29.669200
3,5.602356,2.853367,28.975000,12.504200,28.779900,28.953600


/Users/Admin/Desktop/MSE 446/Title-Generator/.venv/lib/python3.13/site-packages/transformers/generation/utils.py:1616: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Saving final model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./t5-title-generator-final/tokenizer_config.json',
 './t5-title-generator-final/tokenizer.json')

In [8]:
import torch
def load_model():
    model_path = "./t5-title-generator-final"
    
    print(f"Loading model and tokenizer from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Check for Apple Silicon GPU (MPS), fallback to CPU
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")
    else:
        device = torch.device("cpu")
        print("Using CPU")
        
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
    
    return tokenizer, model, device

def generate_title(text, tokenizer, model, device, max_input_length=256):
    # 1. Apply the exact same prefix used during training
    input_text = "generate title: " + text.strip()
    
    # 2. Tokenize the input text
    inputs = tokenizer(
        input_text,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # 3. Generate the title
    # We use beam search (num_beams=4) to get higher quality text generation
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=32,          # Match the target max_length from training
        num_beams=4,                # Explores multiple paths for the best title
        early_stopping=True,        # Stops when it hits an end token
        no_repeat_ngram_size=2,     # Prevents the model from repeating phrases
        repetition_penalty=1.2      # Slightly penalizes repeating words
    )
    
    # 4. Decode the output tokens back into a readable string
    title = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Clean up formatting in case the model outputs weird capitalization
    return title.title()

In [9]:
tokenizer, model, device = load_model()
    
# Test texts from different genres
test_summaries = [
    "In a dystopian future where water is the most precious resource on Earth, a lone courier must transport a hidden cache across the irradiated wasteland. Hunted by warlords and betrayed by his closest ally, he discovers that the cache isn't just water—it's the key to restoring the oceans.",
    
    "After the sudden death of her estranged mother, Sarah inherits a dusty old bookstore in a small coastal town. As she sorts through decades of inventory, she uncovers a series of hidden letters that reveal her mother's secret past and a forbidden romance that changed the town's history forever.",
    
    "This chapter details the economic conditions of the working class in 19th-century London during the height of the Industrial Revolution. It focuses on the harsh realities of the textile mills, child labor laws, and the early formation of trade unions.",
    
    "A young wizard discovers that his magic only works when he is completely terrified. To pass his final exams at the academy, he has to deliberately put himself in increasingly dangerous and absurd situations."
]

print("\n" + "="*50)
for i, summary in enumerate(test_summaries, 1):
    print(f"\n--- Test {i} ---")
    print(f"INPUT TEXT: {summary[:150]}...")
    
    # Generate the title
    title = generate_title(summary, tokenizer, model, device)
    
    print(f"GENERATED TITLE: >> {title} <<")
    
print("\n" + "="*50)

Loading model and tokenizer from ./t5-title-generator-final...
Using Apple Silicon GPU (MPS)


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]



--- Test 1 ---
INPUT TEXT: In a dystopian future where water is the most precious resource on Earth, a lone courier must transport a hidden cache across the irradiated wasteland...


[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED TITLE: >> The Cache <<

--- Test 2 ---
INPUT TEXT: After the sudden death of her estranged mother, Sarah inherits a dusty old bookstore in a small coastal town. As she sorts through decades of inventor...


[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED TITLE: >> The Secrets Of Sarah <<

--- Test 3 ---
INPUT TEXT: This chapter details the economic conditions of the working class in 19th-century London during the height of the Industrial Revolution. It focuses on...


[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED TITLE: >> The Industrial Revolution <<

--- Test 4 ---
INPUT TEXT: A young wizard discovers that his magic only works when he is completely terrified. To pass his final exams at the academy, he has to deliberately put...
GENERATED TITLE: >> The Magician <<

